## Metric Reconciliation Across Pandas, PostgreSql and Power BI

this notebook validates that the metrics and their definitions remain valid across all the tools

In [2]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

orders = pd.read_csv(PROCESSED_DIR / "orders.csv")
items = pd.read_csv(PROCESSED_DIR / "order_items.csv")
returns = pd.read_csv(PROCESSED_DIR / "returns.csv")
shipping = pd.read_csv(PROCESSED_DIR / "shipping.csv")

print("Orders:", len(orders))
print("Items:", len(items))
print("Returns:", len(returns))
print("Shipping:", len(shipping))

Orders: 159988
Items: 315109
Returns: 15908
Shipping: 151223


# Calculate Total orders and net revenue

In [3]:
# Total distinct orders
total_orders = orders["order_id"].nunique()

# Join order lines with order status
sales = items.merge(
    orders[["order_id", "order_status"]],
    on="order_id",
    how="inner",
    validate="many_to_one"
)

sales["gross_amount"] = (
    sales["quantity"] * sales["unit_price"]
)

sales["net_amount"] = (
    sales["gross_amount"] - sales["discount_amount"]
)

sales["is_cancelled"] = (
    sales["order_status"].str.strip().str.lower() == "cancelled"
)

nmv = sales.loc[
    ~sales["is_cancelled"], "net_amount"
].sum()

print("Total Orders:", total_orders)
print("NMV:", nmv)

Total Orders: 159988
NMV: 1199304794.0


## Calculate valid refunds

In [5]:
valid_returns = returns.loc[
    returns["is_valid_return"]
    .astype(str)
    .str.lower()
    .eq("true")
].copy()

refund_amount = pd.to_numeric(
    valid_returns["refund_amount"],
    errors="raise"
).sum()

net_revenue = nmv - refund_amount

print("Refund Amount:", refund_amount)
print("Net Revenue:", net_revenue)

Refund Amount: 58033099
Net Revenue: 1141271695.0


## Calculate Return Rate

In [7]:
valid_return_ids = set(
    valid_returns["order_item_id"]
)

delivered = sales.loc[
    ~sales["is_cancelled"]
    & ~sales["order_status"].str.strip().eq("In Transit")
].copy()

delivered_count = len(delivered)

returned_count = (
    delivered["order_item_id"]
    .isin(valid_return_ids)
    .sum()
)

return_rate = (
    returned_count / delivered_count * 100
    if delivered_count else 0
)

print("Delivered Lines:", delivered_count)
print("Returned Lines:", returned_count)
print("Return Rate:", return_rate, "%")

Delivered Lines: 283531
Returned Lines: 14597
Return Rate: 5.148290663102094 %


## Calculate Shipping fee total

In [9]:
print(
    shipping["order_id"]
    .value_counts()
    .head(10)
)


order_id
O0000001    1
O0000002    1
O0000003    1
O0000004    1
O0000005    1
O0000006    1
O0000007    1
O0000008    1
O0000009    1
O0000010    1
Name: count, dtype: int64


In [10]:
shipping_per_order = (
    shipping.groupby("order_id")["shipping_fee"]
    .max()
)

shipping_total = shipping_per_order.sum()

print("Shipping Fee Total:", shipping_total)

Shipping Fee Total: 81388942.0


In [11]:
python_metrics = {
    "Total orders": total_orders,
    "Net revenue": net_revenue,
    "Refund amount": refund_amount,
    "Return rate": return_rate,
    "Shipping fee total": shipping_total
}

python_results = pd.DataFrame(
    python_metrics.items(),
    columns=["Metric", "Python"]
)

display(python_results)

python_results.to_csv(
    PROJECT_ROOT / "docs" / "python_metrics.csv",
    index=False
)

,Metric,Python
0,Total orders,1.599880e+05
1,Net revenue,1.141272e+09
2,Refund amount,5.803310e+07
3,Return rate,5.148291e+00
4,Shipping fee total,8.138894e+07


In [13]:
import pandas as pd
import numpy as np

# SQL results from PostgreSQL
sql_metrics = {
    "Total orders": 159988,
    "Net revenue": 1141271695.00,
    "Refund amount": 58033099.00,
    "Return rate": 5.1483,
    "Shipping fee total": 81388942.00
}

# Merge with your existing Python results
comparison = python_results.copy()

comparison["SQL"] = comparison["Metric"].map(sql_metrics)

# Calculate difference
comparison["Difference"] = (
    comparison["Python"] - comparison["SQL"]
).abs()

# Compare using appropriate rounding
comparison["Match?"] = np.where(
    comparison.apply(
        lambda row: (
            row["Difference"] <= 0.0001
            if row["Metric"] == "Return rate"
            else row["Difference"] <= 0.01
        ),
        axis=1
    ),
    "MATCH",
    "MISMATCH"
)

# Display without scientific notation
pd.options.display.float_format = "{:,.4f}".format

display(comparison)

# Save report
DOCS_DIR = PROJECT_ROOT / "docs"
DOCS_DIR.mkdir(parents=True, exist_ok=True)

comparison.to_csv(
    DOCS_DIR / "metric_reconciliation.csv",
    index=False
)

,Metric,Python,SQL,Difference,Match?
0,Total orders,"159,988.0000","159,988.0000",0.0000,MATCH
1,Net revenue,"1,141,271,695.0000","1,141,271,695.0000",0.0000,MATCH
2,Refund amount,"58,033,099.0000","58,033,099.0000",0.0000,MATCH
3,Return rate,5.1483,5.1483,0.0000,MATCH
4,Shipping fee total,"81,388,942.0000","81,388,942.0000",0.0000,MATCH
